# 诗歌生成

# 数据处理

In [15]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

# 重要：确保 Eager 模式（TF1/部分混装环境默认可能是图模式）
if hasattr(tf, "executing_eagerly") and not tf.executing_eagerly():
    tf.compat.v1.enable_eager_execution()
print("Eager:", tf.executing_eagerly())

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset(batch_size=32, shuffle=True, max_seq_len=80):
    """
    max_seq_len: 限制序列长度（含 bos/eos），防止长序列导致一次 step 计算量爆炸
    """
    instances, word2id, id2word = process_dataset('poems.txt')
    pad_id = word2id['PAD']

    def batch_iter_fn():
        idx = np.arange(len(instances))
        if shuffle:
            np.random.shuffle(idx)

        for i in range(0, len(idx), batch_size):
            batch = [instances[j] for j in idx[i:i + batch_size]]
            seqs, lens = zip(*batch)
            lens = np.asarray(lens, dtype=np.int32)

            max_len = int(min(lens.max(), max_seq_len))  # 关键：截断最大长度
            x = np.full((len(batch), max_len - 1), pad_id, dtype=np.int32)
            y = np.full((len(batch), max_len - 1), pad_id, dtype=np.int32)
            seqlen = np.minimum(lens, max_len).astype(np.int32) - 1  # 对应截断后的有效长度

            for k, seq in enumerate(seqs):
                seq = np.asarray(seq, dtype=np.int32)
                # seq 原长度为 len(seq)，截断到 max_len
                seq = seq[:max_len]
                L = len(seq) - 1
                if L <= 0:
                    continue
                x[k, :L] = seq[:-1]
                y[k, :L] = seq[1:]

            yield (tf.convert_to_tensor(x),
                   tf.convert_to_tensor(y),
                   tf.convert_to_tensor(seqlen))

    return batch_iter_fn, word2id, id2word

Eager: True


# 模型代码， 完成建模代码

In [16]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        self.embed_layer = tf.keras.layers.Embedding(
            self.v_sz, 64, batch_input_shape=[None, None]
        )

        self.rnncell = tf.keras.layers.SimpleRNNCell(128)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)

    def call(self, inp_ids):
        """
        inp_ids: [b, t]
        return: logits [b, t, v]
        """
        inp_emb = self.embed_layer(inp_ids)   # [b, t, emb]
        h_seq = self.rnn_layer(inp_emb)       # [b, t, h]
        logits = self.dense(h_seq)            # [b, t, v]
        return logits

    def get_next_token(self, x, state):
        """
        x: [b]
        state: list with one tensor [b, h] for SimpleRNNCell
        return: logits [b, v], new_state
        """
        inp_emb = self.embed_layer(x)                 # [b, emb]
        h, new_state = self.rnncell(inp_emb, state)   # h: [b, h], new_state: list
        logits = self.dense(h)                        # [b, v]
        return logits, new_state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [17]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [18]:
def compute_loss(logits, labels, seqlen):
    # logits: [b, t, v], labels: [b, t], seqlen: [b]
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(logits=logits, labels=labels)  # [b, t]
    losses = reduce_avg(losses, seqlen, dim=1)  # [b]
    return tf.reduce_mean(losses)

def train_one_step(model, optimizer, x, y, seqlen):
    """
    完成一步优化过程（显式调用 model.call，绕开 Keras __call__ 的 AutoGraph/trace）
    """
    with tf.GradientTape() as tape:
        logits = model.call(x)  # 关键：不要用 model(x)
        loss = compute_loss(logits, y, seqlen)

    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(epoch, model, optimizer, batch_iterable, print_every=50, max_steps=None):
    loss = 0.0
    for step, (x, y, seqlen) in enumerate(batch_iterable):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % print_every == 0:
            print('epoch', epoch, 'step', step, ': loss', float(loss.numpy()))

        if max_steps is not None and step >= max_steps:
            break
    return loss

# 训练优化过程

In [19]:
optimizer = optimizers.Adam(0.0005)

batch_iter_fn, word2id, id2word = poem_dataset(batch_size=100, shuffle=True)
model = myRNNModel(word2id)

for epoch in range(10):
    loss = train(epoch, model, optimizer, batch_iter_fn(), print_every=50)


epoch 0 step 0 : loss 8.821163177490234
epoch 0 step 50 : loss 6.770924091339111
epoch 0 step 100 : loss 6.545600414276123
epoch 0 step 150 : loss 6.499835014343262
epoch 0 step 200 : loss 6.6115546226501465
epoch 0 step 250 : loss 6.569016933441162
epoch 0 step 300 : loss 6.524219989776611
epoch 0 step 350 : loss 6.58845329284668
epoch 0 step 400 : loss 6.5706987380981445
epoch 1 step 0 : loss 6.5540452003479
epoch 1 step 50 : loss 6.464052677154541
epoch 1 step 100 : loss 6.341538906097412
epoch 1 step 150 : loss 6.21603536605835
epoch 1 step 200 : loss 6.199273586273193
epoch 1 step 250 : loss 6.151977062225342
epoch 1 step 300 : loss 6.121182441711426
epoch 1 step 350 : loss 5.917440891265869
epoch 1 step 400 : loss 5.909659385681152
epoch 2 step 0 : loss 6.030253887176514
epoch 2 step 50 : loss 5.943791389465332
epoch 2 step 100 : loss 5.892157077789307
epoch 2 step 150 : loss 5.9681620597839355
epoch 2 step 200 : loss 5.854340076446533
epoch 2 step 250 : loss 5.807015419006348
ep

# 生成过程

In [42]:
def gen_sentence(max_len=50):
    state = [tf.zeros((1, 128), dtype=tf.float32)]  # SimpleRNNCell: 只有一个 state
    cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    collect = []
    for _ in range(max_len):
        logits, state = model.get_next_token(cur_token, state)
        cur_token = tf.argmax(logits, axis=-1, output_type=tf.int32)
        tid = int(cur_token.numpy()[0])
        if id2word.get(tid) == 'eos':
            break
        collect.append(tid)
    return ''.join(id2word[t] for t in collect)

print(gen_sentence())

一年年日日，春日不堪归。
